# 01_collectAPI_chroniques : niveaux des nappes, station par station

**Objectif** : récupérer, pour chaque station piézométrique de la zone, l'historique de ses mesures de niveau d'eau (les « chroniques »).

- Source : API Hub'Eau « Piézométrie » (https://hubeau.eaufrance.fr/page/api-piezometrie)
- Le projet se fait en trois notebooks :
    1. `01_collectAPI_stations` : liste des stations de la zone (endpoint `/stations`) ; **c'est là que se définit la zone géographique**
    2. **`01_collectAPI_chroniques` (ce notebook)** : téléchargement des mesures de chaque station (endpoint `/chroniques`), **enregistrées à l'état brut**
    3. `03_analyse_visualisation` : contrôle qualité, préparation et analyse des mesures

**Entrée**
- `data/processed/stations_mesurees.csv` : les stations ayant au moins une mesure, produit par `01_collectAPI_stations` (**à exécuter avant**)

**Étapes**
1. Imports
2. Configuration : période à récupérer, paramètres de l'API, chemins
3. Chargement des stations
4. Récupération des chroniques : 1 appel à l'API par station, avec relance des échecs
5. Sauvegarde des données brutes (`data/raw/`)

**Sorties** (`<PERIODE>` = `historique_complet` ou `depuis_AAAA-MM-JJ`, selon la configuration)
- `data/raw/chroniques_<PERIODE>_extrait_le_AAAA-MM-JJ.json` : réponse brute de l'API + contexte de l'extraction → **utilisé par `03_analyse_visualisation`**

#### ========== Prérequis avant d'utiliser le notebook ==========

**Première installation (une seule fois)**
- Cloner le dépôt, puis dans le terminal, à la racine du projet :
  `python3 -m venv .venv` → `source .venv/bin/activate` → `pip install -r requirements.txt`
- Créer le fichier de variables : `cp .env.example .env`

**À chaque session de travail**
- Ouvrir le **dossier racine** `Hub-eau` dans VS Code (pas seulement le notebook)
- Récupérer les dernières modifications : `git pull`
- Vérifier que le **kernel** sélectionné (en haut à droite) est `.venv`
- Exécuter la cellule de vérification ci-dessous

**Pour installer une nouvelle bibliothèque**
- Activer le venv dans le terminal : `source .venv/bin/activate` (l'invite doit commencer par `(.venv)`)
- `pip install <bibliothèque>`, puis `pip freeze > requirements.txt`, puis commit

**Avant chaque commit**
- *Restart Kernel* + *Run All* : le notebook doit tourner en entier sans erreur
- `git status` : `.env`, `.venv/` et les fichiers de `data/` ne doivent jamais apparaître

#### ========== Réutiliser le notebook ==========

Tout se règle dans la cellule **« 2. Configuration »** :

| Je veux… | Variable à modifier |
|---|---|
| tout l'historique de chaque station (par défaut) | `DATE_DEBUT_CHRONIQUES = None` |
| seulement les mesures récentes (ex. compléter une extraction déjà faite) | `DATE_DEBUT_CHRONIQUES = "AAAA-MM-JJ"` |
| utiliser une autre liste de stations | `FICHIER_STATIONS` (section D) |

- **Autre territoire** : modifier la zone dans `01_collectAPI_stations`, l'exécuter, puis exécuter ce notebook sans rien changer.
- Une extraction « depuis une date » crée des fichiers à part : elle n'écrase pas l'historique complet.

In [1]:
# Vérification de l'environnement : le kernel doit être celui du .venv du projet
import sys

print("Python utilisé :", sys.executable)
assert ".venv" in sys.executable, "⚠️ Mauvais kernel : sélectionner le .venv en haut à droite"

Python utilisé : /home/data-08/Documents/00-PROJETS/Hub-eau/.venv/bin/python


## **Appel API - phase B /chroniques**

In [2]:
# 1. Imports des bibliothèques

# Rechargement automatique des modules de src/ quand ils sont modifiés
%load_ext autoreload
%autoreload 2

# Bibliothèque standard (livrée avec Python)
import json                         # sauvegarde des données brutes
import os                           # lecture des variables d'environnement
import sys                          # chemin de recherche des modules
import time                         # pause entre deux stations
from datetime import datetime       # horodatage des fichiers sauvegardés
from pathlib import Path            # chemins de fichiers (data/raw/...)

# Bibliothèques externes (installées dans le .venv, listées dans requirements.txt)
import pandas as pd                 # mise en DataFrame et contrôle des données
import requests                     # uniquement pour intercepter les erreurs réseau (requests.RequestException)
from dotenv import load_dotenv      # chargement du fichier .env

# Modules locaux (dossier src/ à la racine du projet)
sys.path.append(str(Path.cwd().parent))     # rend la racine du projet visible pour Python
from src.hubeau import appeler_api

In [3]:
# 2. Configuration

# --- A. Variables d'environnement (fichier .env) ---
load_dotenv()                                   # cherche le .env en remontant les dossiers

BASE_URL = os.getenv("HUBEAU_BASE_URL")
if BASE_URL is None:
    raise ValueError("HUBEAU_BASE_URL introuvable : vérifier que le fichier .env existe à la racine du projet")

URL_CHRONIQUES = f"{BASE_URL}/chroniques"       # mesures de niveau d'eau par station

# --- B. Période à récupérer ---
# None         => tout l'historique disponible (par défaut)
# "AAAA-MM-JJ" => uniquement les mesures à partir de cette date
DATE_DEBUT_CHRONIQUES = None

# Libellé ajouté aux noms des fichiers de sortie
PERIODE = f"depuis_{DATE_DEBUT_CHRONIQUES}" if DATE_DEBUT_CHRONIQUES else "historique_complet"

# --- C. Paramètres techniques de l'API ---
PAGE_SIZE_CHRONIQUES = 20000                    # maximum autorisé : au-delà de 20 000 mesures, une station serait tronquée
TIMEOUT_S = 30                                  # délai max d'attente d'une réponse (secondes)
PAUSE_S = 0.2                                   # pause entre deux appels, pour ne pas surcharger l'API

# --- D. Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)     # créés s'ils n'existent pas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FICHIER_STATIONS = DATA_PROCESSED / "stations_mesurees.csv"   # produit par 01_collectAPI_stations

# --- Récapitulatif ---
print("URL chroniques   :", URL_CHRONIQUES)
print("Période          :", f"depuis le {DATE_DEBUT_CHRONIQUES}" if DATE_DEBUT_CHRONIQUES else "tout l'historique disponible")
print("Racine projet    :", PROJECT_ROOT)
print("Fichier stations :", FICHIER_STATIONS)

URL chroniques   : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/chroniques
Période          : tout l'historique disponible
Racine projet    : /home/data-08/Documents/00-PROJETS/Hub-eau
Fichier stations : /home/data-08/Documents/00-PROJETS/Hub-eau/data/processed/stations_mesurees.csv


#### 3. Chargement des stations

On lit le fichier produit par `01_collectAPI_stations` (pas de nouvel appel à `/stations`).
Un CSV ne conserve pas les types : les colonnes de dates sont reconverties avec `parse_dates`.

Si une date de début est choisie, on repère aussi les stations **arrêtées avant cette date** : il est normal qu'elles ne renvoient aucune mesure.

In [4]:
# 3. Chargement des stations avec mesures (sortie du notebook 01_collectAPI_stations)

if not FICHIER_STATIONS.exists():
    raise FileNotFoundError(f"{FICHIER_STATIONS} introuvable : exécuter d'abord le notebook 01_collectAPI_stations")

df_stations_mesurees = pd.read_csv(FICHIER_STATIONS,
                                   parse_dates=["date_debut_mesure", "date_fin_mesure"])

codes_stations = df_stations_mesurees["code_bss"].tolist()

print(f"{len(codes_stations)} stations à interroger")
df_stations_mesurees[["code_bss", "nom_commune", "date_debut_mesure",
                      "date_fin_mesure", "nb_mesures_piezo"]].head()

60 stations à interroger


,code_bss,nom_commune,date_debut_mesure,date_fin_mesure,nb_mesures_piezo
0,08422X0190/F1,Loriol-sur-Drôme,1999-12-17,2026-11-13,9157
1,07948X0047/F1,Romans-sur-Isère,2009-06-12,2026-11-13,6136
2,08664X0069/F1,Soyans,2008-12-03,2026-11-11,6526
3,07703X0064/P1,Anneyron,2009-06-09,2026-04-20,6155
4,08187X0238/P1,Beaumont-lès-Valence,2009-06-26,2026-11-13,6343


In [5]:
# Stations arrêtées avant la période choisie => 0 mesure attendue (ce n'est pas une erreur)
# Historique complet : aucune station concernée

if DATE_DEBUT_CHRONIQUES:
    arretees = df_stations_mesurees[df_stations_mesurees["date_fin_mesure"] < DATE_DEBUT_CHRONIQUES]
else:
    arretees = df_stations_mesurees.iloc[0:0]      # tableau vide, mêmes colonnes

print(f"{len(arretees)} station(s) arrêtée(s) avant la période")
arretees[["code_bss", "nom_commune", "date_fin_mesure"]]

0 station(s) arrêtée(s) avant la période


,code_bss,nom_commune,date_fin_mesure


#### 4. Récupération des chroniques : 1 appel par station

L'API renvoie au maximum **20 000 mesures par requête**. Toutes stations confondues, il y en a beaucoup plus : on interroge donc **une station à la fois** et on cumule les résultats dans une seule liste.

- **Erreurs** : une station en échec (API ou réseau) est notée dans `erreurs`, et la boucle continue. Les autres erreurs (bug de code) arrêtent l'exécution, volontairement.
- **Relance** : l'API est parfois lente (délai dépassé). Les stations en échec sont retentées une fois, avec un délai d'attente triplé.
- **Durée** : quelques minutes pour l'historique complet.

In [6]:
# 4a. Fonction de récupération : interroge /chroniques pour une liste de stations

def recuperer_chroniques(codes, timeout_s):
    '''Renvoie (mesures, echecs) : la liste des mesures et la liste des stations en échec.'''
    mesures, echecs = [], []

    for i, code in enumerate(codes, start=1):
        print(f"[{i}/{len(codes)}] {code} : ", end="")

        params_chroniques = {
            "code_bss": code,
            "size": PAGE_SIZE_CHRONIQUES,
            "format": "json",
        }
        if DATE_DEBUT_CHRONIQUES:           # filtre de date seulement s'il est demandé
            params_chroniques["date_debut_mesure"] = DATE_DEBUT_CHRONIQUES

        try:
            mesures.extend(appeler_api(URL_CHRONIQUES, params_chroniques,
                                       timeout_s=timeout_s, pause_s=PAUSE_S))
        except (RuntimeError, requests.RequestException) as e:
            echecs.append({"code_bss": code, "erreur": str(e)})
            print(f"⚠️ Échec : {e}")

        time.sleep(PAUSE_S)  # pause entre deux stations

    return mesures, echecs

In [7]:
# 4b. Récupération des chroniques, station par station

chroniques, erreurs = recuperer_chroniques(codes_stations, timeout_s=TIMEOUT_S)

print(f"\n✅ {len(chroniques)} mesures récupérées pour {len(codes_stations)} stations")
print(f"⚠️ {len(erreurs)} station(s) en échec")

[1/60] 08422X0190/F1 : Page 1 : 9157 résultats (9157/9157)
[2/60] 07948X0047/F1 : Page 1 : 6136 résultats (6136/6136)
[3/60] 08664X0069/F1 : Page 1 : 6526 résultats (6526/6526)
[4/60] 07703X0064/P1 : Page 1 : 6155 résultats (6155/6155)
[5/60] 08187X0238/P1 : Page 1 : 6343 résultats (6343/6343)
[6/60] 07955X0099/P1 : Page 1 : 7450 résultats (7450/7450)
[7/60] 08184X0084/PZ1 : Page 1 : 9637 résultats (9637/9637)
[8/60] 08422X0191/F2 : ⚠️ Échec : HTTPSConnectionPool(host='hubeau.eaufrance.fr', port=443): Read timed out. (read timeout=30)
[9/60] 08424X0006/F2 : Page 1 : 8900 résultats (8900/8900)
[10/60] 08426X0133/P2 : Page 1 : 6249 résultats (6249/6249)
[11/60] 07955X0101/PZ2 : Page 1 : 5908 résultats (5908/5908)
[12/60] 07703X0069/F83 : Page 1 : 6228 résultats (6228/6228)
[13/60] 07725X0071/F3 : Page 1 : 5560 résultats (5560/5560)
[14/60] 08413X0091/F3 : Page 1 : 7126 résultats (7126/7126)
[15/60] 07475X0008/F3 : Page 1 : 8455 résultats (8455/8455)
[16/60] 07476X0034/PZF3 : Page 1 : 152

In [8]:
# 4c. Relance des stations en échec (une seule fois, délai d'attente triplé)

if erreurs:
    a_relancer = [e["code_bss"] for e in erreurs]
    print(f"Nouvelle tentative pour {len(a_relancer)} station(s)\n")
    mesures_relance, erreurs = recuperer_chroniques(a_relancer, timeout_s=TIMEOUT_S * 3)
    chroniques.extend(mesures_relance)

print(f"\n✅ Total : {len(chroniques)} mesures")
print(f"⚠️ {len(erreurs)} station(s) toujours en échec")
erreurs     # liste vide = tout s'est bien passé

Nouvelle tentative pour 2 station(s)

[1/2] 08422X0191/F2 : Page 1 : 8469 résultats (8469/8469)
[2/2] 08184X0069/S : Page 1 : 1453 résultats (1453/1453)

✅ Total : 344995 mesures
⚠️ 0 station(s) toujours en échec


[]

#### 5. Sauvegarde des données brutes

Réponse de l'API **telle quelle**, avec le contexte de l'extraction (date, paramètres, stations, erreurs) pour pouvoir la retracer.
Le fichier est volumineux (~200 Mo pour l'historique complet) : il est enregistré **sans indentation**.

In [9]:
# 5. Sauvegarde des données brutes (data/raw/)

date_extraction = datetime.now()
fichier_brut = DATA_RAW / f"chroniques_{PERIODE}_extrait_le_{date_extraction:%Y-%m-%d}.json"

contenu_brut = {
    "date_extraction": date_extraction.isoformat(timespec="seconds"),
    "url": URL_CHRONIQUES,
    "params": {                                  # paramètres communs à tous les appels
        "date_debut_mesure": DATE_DEBUT_CHRONIQUES,   # None = pas de filtre (historique complet)
        "size": PAGE_SIZE_CHRONIQUES,
        "format": "json",
    },
    "stations_interrogees": codes_stations,
    "erreurs": erreurs,
    "nb_resultats": len(chroniques),
    "data": chroniques,
}

with open(fichier_brut, "w", encoding="utf-8") as f:
    json.dump(contenu_brut, f, ensure_ascii=False)     # sans indent : fichier plus léger

taille_mo = fichier_brut.stat().st_size / 1_000_000
print(f"✅ {len(chroniques)} mesures sauvegardées dans : {fichier_brut} ({taille_mo:.1f} Mo)")

✅ 344995 mesures sauvegardées dans : /home/data-08/Documents/00-PROJETS/Hub-eau/data/raw/chroniques_historique_complet_extrait_le_2026-11-16.json (201.5 Mo)


In [10]:
# Vérification : relecture du fichier sauvegardé
with open(fichier_brut, encoding="utf-8") as f:
    relu = json.load(f)

print("Date d'extraction :", relu["date_extraction"])
print("Nombre de mesures :", len(relu["data"]))
assert len(relu["data"]) == len(chroniques), "⚠️ Le fichier relu ne correspond pas aux données récupérées"

Date d'extraction : 2026-11-16T11:08:18
Nombre de mesures : 344995
